# Install libraries

In [7]:
!pip install pandas numpy yfinance textblob ta matplotlib gradio deep-translator gtts pydub

# Imports libraries

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
from typing import Dict, Optional, Any
from textblob import TextBlob
import ta
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# ---------- Fast TTS (gTTS) ----------
from gtts import gTTS
import os
import tempfile
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='en', target='hi')

# Uper diye gaye code mein jo libraries import ki gayi hain, unka work ye hai:

**pandas - Data manipulation aur analysis ke liye DataFrame provide karta hai.**

**numpy - Numerical operations aur array processing ke liye.**

**yfinance - Yahoo Finance se stock/market data download karne ke liye.**

**datetime, timedelta - Date aur time ke calculations aur manipulations ke liye.**

**typing - Function signatures mein type hints define karne ke liye.**

**textblob - Text data ka sentiment analysis (positive/negative/neutral) karne ke liye.**

**ta - Technical indicators (RSI, MACD, etc.) calculate karne ke liye.**

**matplotlib.pyplot - Charts aur graphs plot karne ke liye.**

**io.BytesIO - In-memory binary data stream handle karne ke liye (images/files ke liye).**

**base64 - Binary data ko ASCII string mein encode/decode karne ke liye (e.g., image embedding).**

**gradio - Machine learning models ke liye quick web UI banane ke liye.**

**warnings - Warning messages ko control ya suppress karne ke liye.**

**gtts - Text-to-speech (Google TTS) se audio generate karne ke liye.**

**os, tempfile - Operating system interactions aur temporary files create/delete karne ke liye.**

**deep_translator - Text ko ek language se doosri language mein translate karne ke liye (yahan English to Hindi).**



# Configuration

In [3]:

class TradingConfig:
    DATA_RANGE = "6mo"
    SAMPLING_INTERVAL = "1d"
    RISKLESS_RATE = 0.02
    NEWS_LOOKBACK_DAYS = 5

**Yeh class trading ke liye configuration parameters set kar rahi hai: 6 mahine ka daily data, 2% risk-free rate, aur news ke liye 5 day lookback.**

# All analysis functions

In [4]:

def fetch_market_quotes(symbol: str, duration: str = "6mo", freq: str = "1d") -> pd.DataFrame:
    ticker_obj = yf.Ticker(symbol)
    data = ticker_obj.history(period=duration, interval=freq)
    if data.empty:
        raise ValueError(f"No price data found for {symbol}")
    return data

def analyze_news_pulse(symbol: str, days_back: int = 5) -> Dict:
    stock = yf.Ticker(symbol)
    articles = stock.news
    if not articles:
        return {
            "polarity": 0.0, "buzz_level": 0.0, "geo_threat": 0.0,
            "verdict": "NEUTRAL", "confidence": 0.3
        }
    cutoff = datetime.now() - timedelta(days=days_back)
    polarity_scores = []
    geo_keywords = ["war", "conflict", "tariff", "sanction", "trade", "crisis", "inflation"]
    buzz_keywords = ["earnings", "dividend", "buyback", "split", "upgrade", "downgrade"]
    geo_risk_value = 0
    buzz_accumulator = 0
    for item in articles[:15]:
        try:
            pub_time = datetime.fromtimestamp(item.get('providerPublishTime', 0))
            if pub_time < cutoff:
                continue
            headline = item.get('title', '')
            blob = TextBlob(headline)
            polarity_scores.append(blob.sentiment.polarity)
            for kw in geo_keywords:
                if kw in headline.lower():
                    geo_risk_value += 0.2
            for kw in buzz_keywords:
                if kw in headline.lower():
                    buzz_accumulator += 0.25
        except:
            continue
    if not polarity_scores:
        return {
            "polarity": 0.0, "buzz_level": 0.0, "geo_threat": 0.0,
            "verdict": "NEUTRAL", "confidence": 0.3
        }
    avg_polarity = np.mean(polarity_scores)
    geo_risk_capped = min(geo_risk_value, 1.0)
    buzz_capped = min(buzz_accumulator, 1.0)
    net_score = avg_polarity + (buzz_capped * 0.3) - (geo_risk_capped * 0.5)
    if net_score > 0.2:
        final_verdict = "BUY"
        confidence_val = min(0.6 + net_score, 0.9)
    elif net_score < -0.2:
        final_verdict = "SELL"
        confidence_val = min(0.6 + abs(net_score), 0.9)
    else:
        final_verdict = "NEUTRAL"
        confidence_val = 0.5
    return {
        "polarity": round(avg_polarity, 3),
        "buzz_level": buzz_capped,
        "geo_threat": round(geo_risk_capped, 2),
        "verdict": final_verdict,
        "confidence": round(confidence_val, 2)
    }

def compute_risk_indicators(price_frame: pd.DataFrame) -> Dict:
    closing = price_frame['Close']
    daily_returns = closing.pct_change().dropna()
    var_95 = np.percentile(daily_returns, 5)
    cvar_95 = daily_returns[daily_returns <= var_95].mean()
    negative_returns = daily_returns[daily_returns < 0]
    downside_std = negative_returns.std() if len(negative_returns) > 0 else daily_returns.std()
    sortino_ratio = (daily_returns.mean() - TradingConfig.RISKLESS_RATE/252) / downside_std if downside_std != 0 else 0
    cumulative = closing / closing.iloc[0]
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_drawdown = drawdown.min()
    annual_return = (closing.iloc[-1] / closing.iloc[0]) ** (252/len(closing)) - 1
    calmar_ratio = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
    vol_annual = daily_returns.std() * np.sqrt(252)
    if vol_annual < 0.2 and max_drawdown > -0.25:
        risk_grade = "LOW"
        agent_signal = "BUY"
        agent_conf = 0.75
    elif vol_annual > 0.4 or max_drawdown < -0.4:
        risk_grade = "HIGH"
        agent_signal = "SELL"
        agent_conf = 0.7
    else:
        risk_grade = "MEDIUM"
        agent_signal = "NEUTRAL"
        agent_conf = 0.5
    return {
        "VaR_95": round(var_95*100, 2), "CVaR_95": round(cvar_95*100, 2),
        "Sortino": round(sortino_ratio, 2), "Calmar": round(calmar_ratio, 2),
        "Volatility_annual": round(vol_annual*100, 2), "Max_Drawdown": round(max_drawdown*100, 2),
        "risk_grade": risk_grade, "agent_signal": agent_signal, "agent_confidence": agent_conf
    }

def evaluate_technical_trends(price_frame: pd.DataFrame) -> Dict:
    close = price_frame['Close']
    high = price_frame['High']
    low = price_frame['Low']
    volume = price_frame['Volume']
    stoch_osc = ta.momentum.StochasticOscillator(high, low, close, window=14, smooth_window=3)
    stoch_k_val = stoch_osc.stoch().iloc[-1]
    stoch_d_val = stoch_osc.stoch_signal().iloc[-1]
    williams_r = ta.momentum.WilliamsRIndicator(high, low, close, lbp=14).williams_r().iloc[-1]
    atr = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range().iloc[-1]
    atr_percent = (atr / close.iloc[-1]) * 100
    sma_short = ta.trend.SMAIndicator(close, window=10).sma_indicator().iloc[-1]
    sma_long = ta.trend.SMAIndicator(close, window=30).sma_indicator().iloc[-1]
    ema_mid = ta.trend.EMAIndicator(close, window=20).ema_indicator().iloc[-1]
    current_price = close.iloc[-1]
    tech_score = 0.0
    if stoch_k_val < 20 and stoch_d_val < 20:
        tech_score += 0.3
    elif stoch_k_val > 80 and stoch_d_val > 80:
        tech_score -= 0.3
    if williams_r < -80:
        tech_score += 0.2
    elif williams_r > -20:
        tech_score -= 0.2
    if sma_short > sma_long and current_price > ema_mid:
        tech_score += 0.25
    elif sma_short < sma_long and current_price < ema_mid:
        tech_score -= 0.25
    if atr_percent > 5:
        tech_score -= 0.2
    avg_vol_20 = volume.iloc[-20:].mean()
    if volume.iloc[-1] > 1.5 * avg_vol_20 and tech_score > 0:
        tech_score += 0.15
    elif volume.iloc[-1] < 0.7 * avg_vol_20 and tech_score < 0:
        tech_score += 0.1
    if tech_score > 0.4:
        tech_signal = "BUY"
        tech_conf = min(0.6 + tech_score*0.4, 0.95)
    elif tech_score < -0.4:
        tech_signal = "SELL"
        tech_conf = min(0.6 + abs(tech_score)*0.4, 0.95)
    else:
        tech_signal = "NEUTRAL"
        tech_conf = 0.5
    return {
        "stoch_k": round(stoch_k_val, 1), "williams_r": round(williams_r, 1),
        "atr_percent": round(atr_percent, 2), "sma_10": round(sma_short, 2),
        "sma_30": round(sma_long, 2), "price": round(current_price, 2),
        "tech_score": round(tech_score, 2), "tech_signal": tech_signal,
        "tech_confidence": round(tech_conf, 2)
    }

def inspect_volume_patterns(price_frame: pd.DataFrame) -> Dict:
    close = price_frame['Close']
    volume = price_frame['Volume']
    vol_ma5 = volume.rolling(5).mean().iloc[-1]
    vol_ma20 = volume.rolling(20).mean().iloc[-1]
    vol_ratio = vol_ma5 / vol_ma20 if vol_ma20 != 0 else 1
    price_returns = close.pct_change().iloc[-10:]
    vol_returns = volume.pct_change().iloc[-10:]
    corr_val = price_returns.corr(vol_returns) if len(price_returns) > 1 else 0
    obv = ta.volume.OnBalanceVolumeIndicator(close, volume).on_balance_volume()
    obv_change = obv.iloc[-1] - obv.iloc[-10] if len(obv) >= 10 else 0
    if vol_ratio > 1.2 and corr_val > 0.3 and obv_change > 0:
        vol_signal = "BUY"
        vol_conf = 0.7
    elif vol_ratio < 0.8 and corr_val < -0.2 and obv_change < 0:
        vol_signal = "SELL"
        vol_conf = 0.7
    else:
        vol_signal = "NEUTRAL"
        vol_conf = 0.5
    return {
        "vol_ratio": round(vol_ratio, 2), "price_vol_corr": round(corr_val, 2),
        "obv_momentum": round(obv_change, 0), "vol_signal": vol_signal,
        "vol_confidence": vol_conf
    }

# Ye code ek stock market analysis system hai jo yfinance se price data fetch karta hai. Fir 4 alag-alag modules run karta hai:

**News sentiment analysis - headlines ka polarity, geo-risk aur buzz score nikalta hai.**

**Risk indicators - VaR, CVaR, Sortino, Calmar, max drawdown calculate karta hai.**

**Technical trends - Stochastic, Williams %R, ATR, SMA crossovers se BUY/SELL signal banata hai.**

**Volume patterns - volume ratio, price-volume correlation aur OBV momentum check karta hai.**

**Final outcome: Har module apna signal (BUY/SELL/NEUTRAL) aur confidence score dictionary mein return karta hai, jise combine karke final trading decision liya ja sakta hai.**

# Fast fusion (no LangGraph)

In [5]:

def generate_full_report(symbol: str, period: str) -> tuple:
    """Returns (report_text, chart_base64)"""
    # Fetch data once
    df = fetch_market_quotes(symbol, period, "1d")

    # Run all agents sequentially (fast enough)
    tech = evaluate_technical_trends(df)
    sent = analyze_news_pulse(symbol, TradingConfig.NEWS_LOOKBACK_DAYS)
    risk = compute_risk_indicators(df)
    vol = inspect_volume_patterns(df)

    # Fusion logic (same as before)
    vol_risk = risk.get('Volatility_annual', 20)
    risk_weight = 0.25 if vol_risk < 30 else 0.4
    tech_weight = 0.35
    sent_weight = 0.2 if sent.get('confidence', 0.5) > 0.5 else 0.15
    vol_weight = 0.2
    total = tech_weight + sent_weight + risk_weight + vol_weight
    w_tech = tech_weight / total
    w_sent = sent_weight / total
    w_risk = risk_weight / total
    w_vol = vol_weight / total

    def map_signal(sig, conf):
        if sig == "BUY":
            return 1 * conf
        elif sig == "SELL":
            return -1 * conf
        return 0

    score = (map_signal(tech.get('tech_signal', 'NEUTRAL'), tech.get('tech_confidence', 0.5)) * w_tech +
             map_signal(sent.get('verdict', 'NEUTRAL'), sent.get('confidence', 0.5)) * w_sent +
             map_signal(risk.get('agent_signal', 'NEUTRAL'), risk.get('agent_confidence', 0.5)) * w_risk +
             map_signal(vol.get('vol_signal', 'NEUTRAL'), vol.get('vol_confidence', 0.5)) * w_vol)

    if score > 0.2:
        final_call = "BUY"
        trust = min(0.5 + score, 0.95)
    elif score < -0.2:
        final_call = "SELL"
        trust = min(0.5 + abs(score), 0.95)
    else:
        final_call = "HOLD"
        trust = 0.5

    report_lines = [
        f"\n MARKET ANALYSIS REPORT",
        f"Symbol: {symbol} | Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "\n--- TECHNICAL SCANNER ---",
        f"Score: {tech.get('tech_score',0)} | Signal: {tech.get('tech_signal','N/A')} (conf {tech.get('tech_confidence',0)})",
        f"Stoch K: {tech.get('stoch_k','N/A')} | Williams%R: {tech.get('williams_r','N/A')}",
        f"Price: {tech.get('price','N/A')} | SMA10/30: {tech.get('sma_10','N/A')}/{tech.get('sma_30','N/A')}",
        "\n--- NEWS SENTIMENT & BUZZ ---",
        f"Polarity: {sent.get('polarity',0)} | Buzz: {sent.get('buzz_level',0)} | Geo Risk: {sent.get('geo_threat',0)} | Signal: {sent.get('verdict','N/A')}",
        "\n--- RISK METRICS ---",
        f"VaR(95%): {risk.get('VaR_95',0)}% | CVaR: {risk.get('CVaR_95',0)}%",
        f"Sortino: {risk.get('Sortino',0)} | Calmar: {risk.get('Calmar',0)}",
        f"Volatility: {risk.get('Volatility_annual',0)}% | Max DD: {risk.get('Max_Drawdown',0)}%",
        f"Risk Grade: {risk.get('risk_grade','N/A')}",
        "\n--- VOLUME TREND ---",
        f"Vol Ratio(5/20): {vol.get('vol_ratio',0)} | Price-Vol Corr: {vol.get('price_vol_corr',0)}",
        f"OBV Momentum: {vol.get('obv_momentum',0)} | Signal: {vol.get('vol_signal','N/A')}",
        f"\n FINAL RECOMMENDATION: {final_call} (Confidence: {trust:.2f})\n"
    ]
    full_report = "\n".join(report_lines)
     # Generate chart
    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(df.index, df['Close'], label='Close Price', color='blue')
    ax.plot(df.index, df['Close'].rolling(20).mean(), label='SMA20', linestyle='--', color='orange')
    ax.set_title(f"{symbol} – Price History")
    ax.legend()
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    encoded_img = base64.b64encode(buf.read()).decode()
    plt.close(fig)

    return full_report, encoded_img

# Ye function ek complete stock analysis report banata hai - jisme technical, news sentiment, risk, aur volume sab kuch mix hota hai.

**1. fetch_market_quotes() se price data aata hai**

 **2. Chaar analysis functions call hote hain:**

     - evaluate_technical_trends
     - analyze_news_pulse
     - compute_risk_indicators
     - inspect_volume_patterns

**3. Fusion logic  har agent ka signal (BUY=+1, SELL=-1, NEUTRAL=0)  weight ke saath multiply karta hai.**

 **Weights dynamic hain:**

   Volatility < 30 → risk_weight = 0.25 (kam)
   Volatility ≥ 30 → risk_weight = 0.4 (zyada)

**4. Weighted sum se final score calculate hota hai**

**5. Score ke hisa se final call:**

   Score > 0.2 → BUY
   Score < -0.2 → SELL
   Else → HOLD

**6. Ek text report build hoti hai (technical, news, risk, volume, final recommendation)**

**7. Matplotlib se price aur SMA20 ka chart banata hai, usse base64 encode karta hai**

**8. Return karta hai (full_report, encoded_img)**

# Fast bilingual TTS (gTTS)

In [6]:

def text_to_speech_bilingual(text_en: str, output_file="bilingual_speech.mp3"):
    """Generate English + Hindi speech using gTTS (much faster than Bark)"""
    # Translate to Hindi
    try:
        text_hi = translator.translate(text_en[:1000])  # limit length to avoid timeout
    except:
        text_hi = "Hindi translation failed."

    # Create temporary files
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_en:
        tmp_en_path = tmp_en.name
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_hi:
        tmp_hi_path = tmp_hi.name

    # Generate English speech
    tts_en = gTTS(text=text_en[:1000], lang='en', slow=False)
    tts_en.save(tmp_en_path)

    # Generate Hindi speech
    tts_hi = gTTS(text=text_hi, lang='hi', slow=False)
    tts_hi.save(tmp_hi_path)

    # Combine both MP3s (simple concatenation works for MP3)
    import subprocess
    # Use ffmpeg if available, else just return first file? Better to combine.
    # Since gradio Audio component can play single file, we'll return English + Hindi together.
    # We can simply return the English file and let user hear English; but requirement is bilingual.
    # We'll combine using pydub (requires ffmpeg). For simplicity, we return both as separate? Not possible.
    # Alternative: return English file and provide Hindi text? No.
    # Best: use pydub if installed, else fallback to English only.
    try:
        from pydub import AudioSegment
        audio_en = AudioSegment.from_mp3(tmp_en_path)
        audio_hi = AudioSegment.from_mp3(tmp_hi_path)
        combined = audio_en + AudioSegment.silent(duration=500) + audio_hi
        combined.export(output_file, format="mp3")
        return output_file
    except ImportError:
        # pydub not installed, just return English (fast fallback)
        return tmp_en_path
    finally:
        # Cleanup temp files
        for p in [tmp_en_path, tmp_hi_path]:
            try: os.unlink(p)
            except: pass

**Ye text_to_speech_bilingual function English text ko Hindi mein translate karta hai aur gTTS se dono languages ki audio banata hai. Phir pydub library se dono audios ko combine karta hai - English, phir 500ms silence, phir Hindi - aur ek final MP3 file export karta hai.**

**Agar pydub missing hai, to sirf English audio ka path return karta hai, lekin wahaan ek bug hai - kyunki temporary file delete ho jaati hai, to returned path useless ho jaata hai. Production mein isko handle karna padega.**




# Gradio interface

In [7]:

def analyze_and_speak(symbol: str, period: str):
    if not symbol.strip():
        return "Please enter a symbol", "", None
    try:
        report_text, chart_b64 = generate_full_report(symbol.upper(), period)
        chart_html = f'<img src="data:image/png;base64,{chart_b64}" style="max-width:100%;">'

        # Generate bilingual speech (fast)
        # Limit report length to ~500 chars for speed
        short_report = report_text[:800]
        audio_path = text_to_speech_bilingual(short_report, "output_speech.mp3")

        return report_text, chart_html, audio_path
    except Exception as e:
        return f"Error: {str(e)}", "", None

with gr.Blocks(title="Fast Bilingual Stock Analyzer") as demo:
    gr.Markdown("# Fast AI Stock Analyzer +  Bilingual Voice (English+Hindi)")
    gr.Markdown("Enter stock symbol, choose period, get analysis and speech within seconds.")
    with gr.Row():
        symbol_inp = gr.Textbox(label="Stock Symbol", value="AAPL")
        period_inp = gr.Dropdown(label="Data Period", choices=["1mo","3mo","6mo","1y"], value="6mo")
    btn = gr.Button("Analyze & Speak")
    with gr.Row():
        report_out = gr.Textbox(label="Analysis Report", lines=18)
        chart_out = gr.HTML(label="Price Chart")
    audio_out = gr.Audio(label="Bilingual Speech (English then Hindi)")

    btn.click(analyze_and_speak, [symbol_inp, period_inp], [report_out, chart_out, audio_out])

if __name__ == "__main__":
    demo.launch(debug=True)

# Yeh code ek Gradio web interface hai jo stock analysis functions ko user-friendly banata hai.

**Ye Gradio web interface hai. analyze_and_speak function user ke symbol aur period ke liye generate_full_report call karta hai, jo text report aur chart (base64) return karta hai. Chart HTML image mein convert hota hai. Report ko pehle 800 chars mein truncate kiya jaata hai, phir text_to_speech_bilingual usse English+Hindi audio mein badalta hai. Output: report text, chart HTML, audio file.**

**Interface mein ek textbox (symbol), dropdown (period), aur button hai. Button click hone par Gradio analyze_and_speak call karta hai aur outputs update karta hai. demo.launch(debug=True) server start karta hai.**